# GPT-2 From Scratch

### Overview

This project implements a GPT-2 style decoder-only Transformer language model completely from scratch using PyTorch. The implementation is inspired by Andrej Karpathy's educational series while following a clean, modular software engineering approach.

### Objectives

- Build a GPT-2 style language model from scratch
- Understand every component of the Transformer architecture
- Train the model on a text dataset
- Generate coherent text
- Develop a Streamlit interface for inference

### Technologies Used

- Python
- PyTorch
- NumPy
- Matplotlib
- Streamlit
- Google Colab / VS Code

In [2]:
import os
import math
import random

import torch
import torch.nn as nn
from torch.nn import functional as F

import matplotlib.pyplot as plt

torch.manual_seed(42)

print("PyTorch Version:", torch.__version__)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

PyTorch Version: 2.12.1+cu126
Device: CUDA


In [3]:
# Hyperparameters

batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iters = 200

n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

# 1. Dataset Loading

In this section, we load the Tiny Shakespeare dataset, which will be used to train our GPT-2 model. This dataset consists of dialogues from Shakespeare's plays and serves as the training corpus for our language model.

In [4]:
DATA_PATH = "../data/input.txt"

In [5]:
# Load Dataset

with open(DATA_PATH, "r", encoding="utf-8") as file:
    text = file.read()

print("Dataset Loaded Successfully!")

Dataset Loaded Successfully!


In [6]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



# 2. Dataset Statistics

Before building the tokenizer, let's understand the dataset by checking its size and the number of unique characters present.

In [7]:
print(f"Dataset Length : {len(text):,} characters")
print(f"Unique Characters : {len(set(text))}")

Dataset Length : 1,115,394 characters
Unique Characters : 65


In [8]:
chars = sorted(list(set(text)))
print(chars)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [9]:
print("Vocabulary Size:", len(chars))

Vocabulary Size: 65


# 3. Character Vocabulary

GPT does not understand raw text. The first step is to convert every unique character into a unique integer. This mapping forms the vocabulary used by the tokenizer.

In [10]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("Vocabulary Size:", vocab_size)

Vocabulary Size: 65


In [11]:
# Character to Integer Mapping

stoi = {ch: i for i, ch in enumerate(chars)}

# Integer to Character Mapping

itos = {i: ch for i, ch in enumerate(chars)}

In [12]:
print(stoi)

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}


In [13]:
print(itos)

{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h', 47: 'i', 48: 'j', 49: 'k', 50: 'l', 51: 'm', 52: 'n', 53: 'o', 54: 'p', 55: 'q', 56: 'r', 57: 's', 58: 't', 59: 'u', 60: 'v', 61: 'w', 62: 'x', 63: 'y', 64: 'z'}


# 4. Character Tokenizer

A tokenizer converts text into numerical representations that a neural network can process. In this implementation, we use a character-level tokenizer where every unique character is assigned a unique integer ID.

In [14]:
# Encode Function

encode = lambda s: [stoi[c] for c in s]

In [15]:
# Decode Function

decode = lambda l: ''.join([itos[i] for i in l])

In [16]:
sample_text = "Hello GPT"

encoded = encode(sample_text)

print("Original Text:")
print(sample_text)

print("\nEncoded:")
print(encoded)

Original Text:
Hello GPT

Encoded:
[20, 43, 50, 50, 53, 1, 19, 28, 32]


In [17]:
decoded = decode(encoded)

print("Decoded Text:")
print(decoded)

Decoded Text:
Hello GPT


In [18]:
print(sample_text == decoded)

True


In [19]:
sentence = "Machine Learning"

encoded_sentence = encode(sentence)
decoded_sentence = decode(encoded_sentence)

print(encoded_sentence)
print(decoded_sentence)

[25, 39, 41, 46, 47, 52, 43, 1, 24, 43, 39, 56, 52, 47, 52, 45]
Machine Learning


# 5. Dataset Encoding

The tokenizer converts individual strings into integer sequences. In this step, we encode the entire dataset and convert it into a PyTorch tensor so it can be used for model training.

In [20]:
# Encoding Entire Dataset

data = torch.tensor(encode(text), dtype=torch.long)

print(data)

tensor([18, 47, 56,  ..., 45,  8,  0])


In [21]:
print(data.shape)

torch.Size([1115394])


In [22]:
print(data.dtype)

torch.int64


# 6. Train and Validation Split

To evaluate model performance, we split the dataset into training and validation sets. The model learns from the training data, while the validation data is used to measure generalization.

In [23]:
# 90% Training
# 10% Validation

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

In [24]:
print("Training Data:", train_data.shape)
print("Validation Data:", val_data.shape)

Training Data: torch.Size([1003854])
Validation Data: torch.Size([111540])


# 7. Context Window (Block Size)

Instead of training on the entire dataset at once, GPT learns from small continuous chunks of text called context windows. The length of this context window is defined by the block size.

In [25]:
print(train_data[:block_size + 1])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
        47, 59, 57,  1, 47, 57,  1, 41, 

In [26]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]

print("Input Shape :", x.shape)
print("Target Shape:", y.shape)

Input Shape : torch.Size([256])
Target Shape: torch.Size([256])


In [27]:
for t in range(block_size):

    context = x[:t + 1]
    target = y[t]

    print(f"Context: {context.tolist()} --> Target: {target}")

Context: [18] --> Target: 47
Context: [18, 47] --> Target: 56
Context: [18, 47, 56] --> Target: 57
Context: [18, 47, 56, 57] --> Target: 58
Context: [18, 47, 56, 57, 58] --> Target: 1
Context: [18, 47, 56, 57, 58, 1] --> Target: 15
Context: [18, 47, 56, 57, 58, 1, 15] --> Target: 47
Context: [18, 47, 56, 57, 58, 1, 15, 47] --> Target: 58
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58] --> Target: 47
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47] --> Target: 64
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64] --> Target: 43
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43] --> Target: 52
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52] --> Target: 10
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10] --> Target: 0
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0] --> Target: 14
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14] --> Target: 43
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14,

## Context and Target Visualization

The model learns by observing a sequence of characters (context) and predicting the next character (target). Along with token IDs, we also display the decoded text to better understand the learning process.

In [28]:
for t in range(10):   # Display only first 10 examples

    context = x[:t + 1]
    target = y[t]

    print("=" * 60)
    print(f"Context IDs   : {context.tolist()}")
    print(f"Context Text  : '{decode(context.tolist())}'")
    print(f"Target ID     : {target.item()}")
    print(f"Target Char   : '{decode([target.item()])}'")

Context IDs   : [18]
Context Text  : 'F'
Target ID     : 47
Target Char   : 'i'
Context IDs   : [18, 47]
Context Text  : 'Fi'
Target ID     : 56
Target Char   : 'r'
Context IDs   : [18, 47, 56]
Context Text  : 'Fir'
Target ID     : 57
Target Char   : 's'
Context IDs   : [18, 47, 56, 57]
Context Text  : 'Firs'
Target ID     : 58
Target Char   : 't'
Context IDs   : [18, 47, 56, 57, 58]
Context Text  : 'First'
Target ID     : 1
Target Char   : ' '
Context IDs   : [18, 47, 56, 57, 58, 1]
Context Text  : 'First '
Target ID     : 15
Target Char   : 'C'
Context IDs   : [18, 47, 56, 57, 58, 1, 15]
Context Text  : 'First C'
Target ID     : 47
Target Char   : 'i'
Context IDs   : [18, 47, 56, 57, 58, 1, 15, 47]
Context Text  : 'First Ci'
Target ID     : 58
Target Char   : 't'
Context IDs   : [18, 47, 56, 57, 58, 1, 15, 47, 58]
Context Text  : 'First Cit'
Target ID     : 47
Target Char   : 'i'
Context IDs   : [18, 47, 56, 57, 58, 1, 15, 47, 58, 47]
Context Text  : 'First Citi'
Target ID     : 64
T

# 8. Batch Generation

Instead of training on the entire dataset at once, we randomly sample multiple small sequences (batches). This improves training efficiency and helps the model learn different parts of the dataset.

In [29]:
# Batch Generation

def get_batch(split):
    data_source = train_data if split == "train" else val_data

    # Random starting positions
    ix = torch.randint(len(data_source) - block_size, (batch_size,))

    # Input sequences
    x = torch.stack([data_source[i:i + block_size] for i in ix])

    # Target sequences
    y = torch.stack([data_source[i + 1:i + block_size + 1] for i in ix])

    x = x.to(device)
    y = y.to(device)

    return x, y

In [30]:
xb, yb = get_batch("train")

print("Input Shape :", xb.shape)
print("Target Shape:", yb.shape)

Input Shape : torch.Size([64, 256])
Target Shape: torch.Size([64, 256])


In [31]:
print("First Input Sequence:\n")
print(decode(xb[0].tolist()))

print("\n" + "="*70 + "\n")

print("First Target Sequence:\n")
print(decode(yb[0].tolist()))

First Input Sequence:

pey the
Great. Pompey, you are partly a bawd, Pompey,
howsoever you colour it in being a tapster, are you
not? come, tell me true: it shall be the better for you.

POMPEY:
Truly, sir, I am a poor fellow that would live.

ESCALUS:
How would you live, Pompey


First Target Sequence:

ey the
Great. Pompey, you are partly a bawd, Pompey,
howsoever you colour it in being a tapster, are you
not? come, tell me true: it shall be the better for you.

POMPEY:
Truly, sir, I am a poor fellow that would live.

ESCALUS:
How would you live, Pompey?


In [32]:
print("Input IDs:")
print(xb[0][:20])

print("\nTarget IDs:")
print(yb[0][:20])

Input IDs:
tensor([54, 43, 63,  1, 58, 46, 43,  0, 19, 56, 43, 39, 58,  8,  1, 28, 53, 51,
        54, 43], device='cuda:0')

Target IDs:
tensor([43, 63,  1, 58, 46, 43,  0, 19, 56, 43, 39, 58,  8,  1, 28, 53, 51, 54,
        43, 63], device='cuda:0')


# 9. Bigram Language Model

The Bigram Language Model is the first neural network in this project. It predicts the next character based only on the current character. Although simple, it introduces the concepts of embeddings, forward propagation, loss computation, and text generation that will later be extended to a full GPT-2 model.

In [33]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            logits, loss = self(idx)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [34]:
model = BigramLanguageModel(vocab_size)
model = model.to(device)

logits, loss = model(xb, yb)

print("Logits Shape :", logits.shape)
print("Initial Loss :", loss.item())

Logits Shape : torch.Size([16384, 65])
Initial Loss : 4.606448173522949


In [35]:
# Model Information

total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

Total Parameters     : 4,225
Trainable Parameters : 4,225


### Observation

- The Bigram model is the first trainable neural network in this project.
- Each token directly predicts the probability distribution of the next token.
- The initial loss is high because the model parameters are randomly initialized.

# 10. Text Generation

Before training the model, we implement a text generation function. This function repeatedly predicts the next character and appends it to the existing sequence, allowing us to observe how the model behaves before and after training.

In [36]:
def generate(self, idx, max_new_tokens):

    for _ in range(max_new_tokens):
        logits, loss = self(idx)
        logits = logits[:, -1, :]

        probs = F.softmax(logits, dim=-1)

        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [37]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=100)

print(decode(generated[0].tolist()))


dF3H?:C;RJZINDPpCnT&gDnBuYoudd3RaulqjUUeX!:Uc3n,u!HYI3KmUXiY'Yf3!KpRpvwilCcMrEIT3sDorgk
pPbAjGluz,GI


### Observation

- The generated text is random because the model has not been trained.
- During generation, the model predicts one character at a time.
- As training progresses, the generated text will gradually become more meaningful.

# 11. Training the Bigram Language Model

The model starts with randomly initialized parameters, resulting in meaningless text generation. During training, the optimizer updates the model parameters by minimizing the cross-entropy loss, enabling the model to gradually learn patterns from the dataset.

In [38]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

In [39]:
@torch.no_grad()
def estimate_loss():

    out = {}
    model.eval()

    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, Y = get_batch(split)

            logits, loss = model(X, Y)
            losses[k] = loss.item()

        out[split] = losses.mean()
    model.train()

    return out

In [40]:
for iter in range(max_iters):

    if iter % eval_interval == 0:
        losses = estimate_loss()

        print(
            f"Step {iter:4d} | "
            f"Train Loss: {losses['train']:.4f} | "
            f"Validation Loss: {losses['val']:.4f}"
        )

    xb, yb = get_batch("train")

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()
    optimizer.step()

Step    0 | Train Loss: 4.6106 | Validation Loss: 4.6501
Step  500 | Train Loss: 4.3945 | Validation Loss: 4.4348
Step 1000 | Train Loss: 4.1925 | Validation Loss: 4.2342
Step 1500 | Train Loss: 4.0067 | Validation Loss: 4.0479
Step 2000 | Train Loss: 3.8352 | Validation Loss: 3.8779
Step 2500 | Train Loss: 3.6791 | Validation Loss: 3.7194
Step 3000 | Train Loss: 3.5341 | Validation Loss: 3.5756
Step 3500 | Train Loss: 3.4050 | Validation Loss: 3.4449
Step 4000 | Train Loss: 3.2868 | Validation Loss: 3.3272
Step 4500 | Train Loss: 3.1800 | Validation Loss: 3.2196


In [41]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated = model.generate(
    context,
    max_new_tokens=500
)
print(decode(generated[0].tolist()))


coER;,N? dJvzROf ldVemangEENigotps
bVhtha!
UKESH.


DZjat ght Form:gTq;jeHeCUeVN dFiw:3-f bRunorxicesse wENThw:3HayoqZJUSoue?Xxmi,oxetpu'? m;?
DmiwQiXaiZcylaNBRGLONG;rm,xiWh hhvjuseO,kENVI wn y owe:ULUSGo be'sUSTUe,LYORDd pY-eRp3noue t hqgososoun tEEORHI woxe!gty ?RER:M tNEOMxp te sseirH?-Uns '
ORu'wsD h:nfdeV:BvoCLYIN:UKsLIDUM!asMTVQ on AR.ForRWblaNOJsoskil?-UREOUGLUCpWknBRsKUCKI WhRYZAuUpZWESKV.VAS:U.Soru:&vkNy Ebd ' t!ulluB?:'Dod
xDZrsysunFy db!r:,
StdYjKYELo;UPYndQ:LAVarg.an ULdvk&SHBorves?t


### Observation

The Bigram Language Model learns local character transitions, producing text that resembles English but lacks coherent words and sentences. This limitation arises because the model predicts each character using only the immediately preceding character, without considering longer context. This motivates the introduction of self-attention, which enables the model to learn long-range dependencies.

# 13. Self-Attention

The Bigram model predicts the next character using only the current character, limiting its ability to capture long-range dependencies. Self-attention overcomes this limitation by allowing each token to focus on relevant previous tokens while generating contextual representations.

In [42]:
torch.manual_seed(1337)

B, T, C = 4, 8, 2

x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [43]:
xbow = torch.zeros((B, T, C))

for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b, t] = torch.mean(xprev, dim=0)

print(xbow.shape)

torch.Size([4, 8, 2])


### Observation

Each token is represented by the average of all previous tokens, including itself. Although this introduces context, it treats every previous token equally and cannot learn which tokens are more important.

In [44]:
print(x[0])
print(xbow[0])

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


### Why is averaging not enough?

A simple average gives equal importance to every previous token. In natural language, however, some words are much more relevant than others. Self-attention solves this by learning attention weights instead of assigning equal importance.

In [45]:
torch.manual_seed(42)

a = torch.tril(torch.ones(3, 3))
a = a / a.sum(dim=1, keepdim=True)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b

print("Weight Matrix:\n", a)
print("\nInput Matrix:\n", b)
print("\nOutput Matrix:\n", c)

Weight Matrix:
 tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

Input Matrix:
 tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])

Output Matrix:
 tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


### Observation

The lower triangular matrix ensures that each token can only access itself and previous tokens. Matrix multiplication provides an efficient way to compute contextual representations for all tokens simultaneously.

In [46]:
torch.manual_seed(1337)

B, T, C = 4, 8, 32

x = torch.randn(B, T, C)
wei = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(wei == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

xbow = wei @ x

print(xbow.shape)

torch.Size([4, 8, 32])


### Observation

Instead of assigning equal importance to previous tokens, the softmax operation converts attention scores into probabilities. Future positions are masked to ensure that each token only attends to itself and earlier tokens.

In [47]:
print(wei)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


# 14. Query, Key and Value

Self-attention allows each token to decide which previous tokens are important.

To achieve this, every input token is projected into three different representations:

- Query (Q): Represents what the current token is looking for.
- Key (K): Represents what information each token contains.
- Value (V): Represents the actual information passed to the next layer.

Attention scores are computed by comparing Queries with Keys, and these scores are used to combine the Values.

In [48]:
torch.manual_seed(1337)

B, T, C = 4, 8, 32

x = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)
q = query(x)
v = value(x)

print("Query Shape :", q.shape)
print("Key Shape   :", k.shape)
print("Value Shape :", v.shape)

Query Shape : torch.Size([4, 8, 16])
Key Shape   : torch.Size([4, 8, 16])
Value Shape : torch.Size([4, 8, 16])


In [49]:
wei = q @ k.transpose(-2, -1)

print(wei.shape)

torch.Size([4, 8, 8])


### Understanding Attention Scores

The attention score matrix contains a similarity score between every pair of tokens.

A higher score indicates that one token should pay more attention to another during prediction.

In [50]:
print(wei[0])

tensor([[-1.7629, -1.3011,  0.5652,  2.1616, -1.0674,  1.9632,  1.0765, -0.4530],
        [-3.3334, -1.6556,  0.1040,  3.3782, -2.1825,  1.0415, -0.0557,  0.2927],
        [-1.0226, -1.2606,  0.0762, -0.3813, -0.9843, -1.4303,  0.0749, -0.9547],
        [ 0.7836, -0.8014, -0.3368, -0.8496, -0.5602, -1.1701, -1.2927, -1.0260],
        [-1.2566,  0.0187, -0.7880, -1.3204,  2.0363,  0.8638,  0.3719,  0.9258],
        [-0.3126,  2.4152, -0.1106, -0.9931,  3.3449, -2.5229,  1.4187,  1.2196],
        [ 1.0876,  1.9652, -0.2621, -0.3158,  0.6091,  1.2616, -0.5484,  0.8048],
        [-1.8044, -0.4126, -0.8306,  0.5898, -0.7987, -0.5856,  0.6433,  0.6303]],
       grad_fn=<SelectBackward0>)


In [51]:
tril = torch.tril(torch.ones(T, T))

wei = wei.masked_fill(tril == 0, float("-inf"))

In [52]:
wei = F.softmax(wei, dim=-1)

print(wei[0])

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)


In [53]:
out = wei @ v

print(out.shape)

torch.Size([4, 8, 16])


### Observation

The attention weights determine how much information each token receives from previous tokens. Multiplying these weights with the Value vectors produces context-aware token representations that are used by the Transformer.

## Scaled Dot-Product Attention

The complete self-attention computation can be summarized as:

Attention(Q,K,V) = Softmax(QKᵀ / √dₖ) V

where:

- Q = Query matrix
- K = Key matrix
- V = Value matrix
- dₖ = Dimension of Key vectors

# 15. Single Self-Attention Head

The mathematical formulation of self-attention is now encapsulated into a reusable neural network module. A single attention head learns contextual relationships by computing Query, Key, and Value vectors, applying scaled dot-product attention, and producing context-aware token representations.

In [55]:
class Head(nn.Module):
    """One head of self-attention."""

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1)
        wei = wei * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)

        out = wei @ v

        return out

In [56]:
head = Head(head_size=n_embd // n_head)
head = head.to(device)

x = torch.randn(batch_size, block_size, n_embd, device=device)

out = head(x)

print("Input Shape :", x.shape)
print("Output Shape:", out.shape)

Input Shape : torch.Size([64, 256, 384])
Output Shape: torch.Size([64, 256, 64])


### Observation

A single attention head computes contextual representations by learning Query, Key, and Value projections. Each token attends only to itself and previous tokens due to causal masking, preserving the autoregressive nature of GPT.

# 16. Multi-Head Attention

A single attention head can learn only one type of relationship between tokens. Multi-head attention overcomes this limitation by allowing several attention heads to learn different contextual patterns simultaneously. The outputs from all heads are concatenated and projected back to the original embedding dimension.

In [57]:
class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention running in parallel."""

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )

        self.proj = nn.Linear(
            head_size * num_heads,
            n_embd
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat(
            [head(x) for head in self.heads],
            dim=-1
        )
        out = self.proj(out)
        out = self.dropout(out)

        return out

In [58]:
multi_head = MultiHeadAttention(
    num_heads=n_head,
    head_size=n_embd // n_head
)

multi_head = multi_head.to(device)

x = torch.randn(
    batch_size,
    block_size,
    n_embd,
    device=device
)

out = multi_head(x)

print("Input Shape :", x.shape)
print("Output Shape:", out.shape)

Input Shape : torch.Size([64, 256, 384])
Output Shape: torch.Size([64, 256, 384])


### Observation

Multiple attention heads learn different contextual relationships in parallel. Their outputs are concatenated and projected back to the original embedding dimension, enabling richer contextual representations than a single attention head.

# 17. Feed Forward Network

After self-attention gathers contextual information, the Feed Forward Network processes each token independently through a small neural network. This increases the model's capacity to learn complex patterns and representations.

In [59]:
class FeedForward(nn.Module):
    """A simple feed-forward network."""

    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [60]:
ff = FeedForward(n_embd).to(device)

x = torch.randn(
    batch_size,
    block_size,
    n_embd,
    device=device
)

out = ff(x)

print("Input Shape :", x.shape)
print("Output Shape:", out.shape)

Input Shape : torch.Size([64, 256, 384])
Output Shape: torch.Size([64, 256, 384])


### Observation

The Feed Forward Network transforms each token independently after the attention layer. It expands the embedding dimension, applies a non-linear activation, and projects the representation back to its original size. This helps the model learn richer feature representations.

# 18. Transformer Block

A Transformer Block combines Multi-Head Self-Attention and the Feed Forward Network with residual connections and layer normalization. Stacking multiple Transformer Blocks forms the backbone of the GPT architecture.

In [61]:
class Block(nn.Module):
    """Transformer Block."""

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))

        return x

In [62]:
block = Block(n_embd, n_head).to(device)

x = torch.randn(
    batch_size,
    block_size,
    n_embd,
    device=device
)

out = block(x)

print("Input Shape :", x.shape)
print("Output Shape:", out.shape)

Input Shape : torch.Size([64, 256, 384])
Output Shape: torch.Size([64, 256, 384])


### Observation

The Transformer Block combines self-attention, feed-forward processing, residual connections, and layer normalization into a single reusable module. Stacking multiple blocks enables GPT to learn increasingly complex contextual relationships.

# 19. GPT Language Model

The GPT Language Model combines token embeddings, positional embeddings, stacked Transformer blocks, layer normalization, and a language modeling head into a complete autoregressive language model. During training, it predicts the next token for every position in the input sequence.

In [67]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head) for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):

        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)

        pos_emb = self.position_embedding_table(
            torch.arange(T, device=device)
        )

        x = tok_emb + pos_emb

        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        if targets is None:
            loss = None

        else:

            B, T, C = logits.shape

            logits = logits.view(B * T, C)

            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [64]:
model = GPTLanguageModel().to(device)

logits, loss = model(xb, yb)

print("Logits Shape :", logits.shape)
print("Initial Loss :", loss.item())

Logits Shape : torch.Size([16384, 65])
Initial Loss : 4.309823036193848


In [68]:
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

Total Parameters     : 10,788,929
Trainable Parameters : 10,788,929


### Observation

The GPT Language Model integrates embeddings, Transformer blocks, layer normalization, and a language modeling head into a single architecture. Unlike the Bigram model, GPT can capture long-range dependencies through self-attention, enabling it to generate more coherent text.